# บทที่ 12 - การลดประวัติการแชทด้วย Agent Scratchpad

สมุดบันทึกนี้แสดงวิธีการจัดการบริบทในการสนทนาที่ยาวโดยใช้ Microsoft Agent Framework เมื่อการสนทนายาวขึ้น จำนวนโทเค็นจะเพิ่มขึ้น — และสุดท้ายเกินขนาดหน้าต่างบริบทของโมเดล เราจัดการสิ่งนี้ด้วย **รูปแบบสรุปบริบท** และ **agent scratchpad** สำหรับหน่วยความจำที่ยั่งยืน

## สิ่งที่คุณจะได้เรียนรู้:
1. **ทำไมการจัดการบริบทจึงสำคัญ**: ทำความเข้าใจขีดจำกัดโทเค็นและหน้าต่างบริบท
2. **เอเจนต์ที่มีความรู้บริบท**: การสร้างเอเจนต์ที่จัดการบริบทการสนทนาของตนเอง
3. **รูปแบบสรุปบริบท**: ใช้เครื่องมือในการย่อประวัติการสนทนา
4. **Agent Scratchpad**: หน่วยความจำที่ยั่งยืนที่ยังคงอยู่แม้หลังการลดบริบท

## ข้อกำหนดเบื้องต้น:
- ตั้งค่า Azure OpenAI พร้อมกำหนดตัวแปรสภาพแวดล้อม
- มีความเข้าใจพื้นฐานเกี่ยวกับแนวคิดเอเจนต์จากบทเรียนก่อนหน้า


## การตั้งค่า


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ทำไมการจัดการบริบทถึงสำคัญ

LLM ทุกตัวมี **หน้าต่างบริบท** ที่จำกัด — คือจำนวนโทเค็นสูงสุดที่มันสามารถประมวลผลได้ในคำขอหนึ่งครั้ง เมื่อการสนทนาแบบหลายรอบดำเนินไป:

- **จำนวนโทเค็นเพิ่มขึ้นอย่างเชิงเส้น** กับทุกข้อความของผู้ใช้และการตอบกลับของผู้ช่วย
- **โทเค็นคำสั่งมีผลต่อค่าใช้จ่ายมากที่สุด** เพราะประวัติทั้งหมดจะถูกส่งซ้ำในทุกรอบ
- ในที่สุดการสนทนา **เกินหน้าต่างบริบท** และโมเดลจะตัดข้อความหรือเกิดข้อผิดพลาด

### กลยุทธ์ในการจัดการบริบท

| กลยุทธ์ | วิธีการทำงาน | ข้อแลกเปลี่ยน |
|---|---|---|
| **การตัดทอน** | ลบข้อความเก่าที่สุดออก | สูญเสียบริบทช่วงต้น |
| **การสรุป** | ย่อข้อความเก่าเป็นสรุป | สูญเสียรายละเอียดบางส่วน แต่เก็บประเด็นสำคัญไว้ |
| **สมุดจด / หน่วยความจำภายนอก** | เก็บข้อมูลสำคัญนอกการสนทนา | ต้องเรียกใช้เครื่องมือ แต่รอดพ้นจากการลดขนาดใด ๆ |

ในสมุดบันทึกนี้เรารวม **การสรุป** กับ **เครื่องมือสมุดจด** เพื่อให้เอเย่นต์สามารถรักษาความต่อเนื่องแม้ว่าประวัติการสนทนาจะถูกย่อ


## การสร้างตัวแทนที่ตระหนักถึงบริบท


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## การจำลองการสนทนายาว

ให้เราลองเดินผ่านการสนทนาหลายช่วงเพื่อดูว่าบริบทสะสมอย่างไร ตัวแทนควรเก็บรายละเอียดสำคัญ (ความชอบ งบประมาณ วันที่เดินทาง) ไว้ในแต่ละช่วงและแสดงให้เห็นถึงความต่อเนื่อง


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

สังเกตว่าตัวแทนยังคงเก็บบริบทจากการสนทนาก่อนหน้า — มันรู้เกี่ยวกับญี่ปุ่น ซูชิ วัด การถ่ายภาพ งบประมาณ $3000 การเดินทางคนเดียว และช่วงเวลาประมาณเดือนเมษายน ในการสนทนาแบบสั้น ๆ นี้ทำงานได้ดี แต่เมื่อการสนทนาเพิ่มขึ้น ประวัติทั้งหมดจะมีค่าใช้จ่ายที่สูงขึ้นสำหรับการส่งซ้ำ

มาดำเนินการสนทนาต่อด้วยรอบการพูดคุยมากขึ้นเพื่อดูการสะสมบริบท:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## รูปแบบการสรุปบริบท

ขณะที่การสนทนาเติบโตขึ้น เราสามารถใช้ **เครื่องมือสรุป** เพื่อย่อบริบทที่สะสมไว้ให้อยู่ในรูปแบบที่กะทัดรัด ตัวแทนจะเรียกเครื่องมือนี้เพื่อบันทึกความชอบที่สำคัญ เพื่อให้แม้ว่าข้อความเก่าจะถูกลบ ข้อมูลสำคัญยังคงถูกเก็บรักษาไว้

รูปแบบนี้เป็นบล็อกพื้นฐานสำหรับการลดประวัติที่ซับซ้อนมากขึ้น:
1. ตัวแทนระบุข้อเท็จจริงสำคัญจากการสนทนา
2. เรียกใช้เครื่องมือสรุปเพื่อเก็บไว้
3. ข้อความเก่าสามารถถูกลบอย่างปลอดภัยเพราะสรุปนั้นจับสิ่งที่สำคัญได้

ด้านล่างเรากำหนดเครื่องมือ `summarize_preferences` ที่ตัวแทนสามารถเรียกใช้เพื่อบันทึกสรุปกะทัดรัดของสิ่งที่ได้เรียนรู้


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีจัดการบริบทในการสนทนาของเอเย่นต์ที่ดำเนินงานเป็นเวลานานโดยใช้ Microsoft Agent Framework:

### แนวคิดสำคัญ
- **หน้าต่างบริบทมีขีดจำกัด** — ทุกโทเค็นในประวัติการสนทนามีค่าใช้จ่ายและนับรวมในขีดจำกัด
- **เครื่องมือสรุปเนื้อหา** ช่วยให้เอเย่นต์ย่อบริบทที่สะสมไว้เป็นสรุปสั้น ๆ ลดการใช้โทเค็นขณะคงข้อมูลสำคัญไว้
- **สมุดจดเอเย่นต์** ให้หน่วยความจำภายนอกที่คงอยู่อย่างถาวรซึ่งไม่ถูกลบเมื่อประวัติการสนทนาถูกลดขนาด

### สิ่งที่คุณสร้างขึ้น
- **เอเย่นต์ที่รับรู้บริบท** รักษาความต่อเนื่องระหว่างการสนทนาแบบหลายรอบ
- **เครื่องมือสรุปเนื้อหา** (`summarize_preferences`) ที่บันทึกรายละเอียดผู้ใช้สำคัญในรูปแบบสั้น
- **การสนทนาแบบหลายรอบ** ที่แสดงการเก็บรักษาและจัดการการเปลี่ยนแปลงของบริบท

### การใช้งานในโลกจริง
- **บอทบริการลูกค้า**: จำความชอบของลูกค้าในช่วงเซสชันสนับสนุนยาวนาน
- **ผู้ช่วยส่วนตัว**: ติดตามโครงการที่กำลังดำเนินการโดยไม่ต้องอธิบายบริบทซ้ำ
- **ติวเตอร์การศึกษา**: รักษาความก้าวหน้าของนักเรียนในหลายๆ ปฏิสัมพันธ์

### ขั้นตอนถัดไป
- นำเครื่องมือสมุดจดที่มีการเก็บข้อมูลแบบไฟล์มาใช้
- เพิ่มการตัดประวัติอัตโนมัติหลังจากสรุปเนื้อหา
- ผสานกับฐานข้อมูลเวกเตอร์สำหรับการค้นหาหน่วยความจำเชิงความหมาย
- สร้างเอเย่นต์ที่สามารถสานต่อการสนทนาในอีกหลายวันต่อมาพร้อมบริบทครบถ้วน


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
